# 01 - Mô hình Content-Based Filtering

**Mục tiêu:**
Xây dựng mô hình Gợi ý dựa trên nội dung bài hát bằng cách:
1. **Trích xuất Đặc trưng (Feature Extraction):** Tạo ra đoạn văn bản (content) bằng cách kết hợp thông tin bài hát (tên bài hát lặp 2 lần để tăng trọng số, ca sĩ, thể loại, nhạc sĩ, tác giả, ngôn ngữ).
2. **K-Nearest Neighbors (KNN):** Tính toán khoảng cách Cosine giữa các vector TF-IDF để tìm ra các bài hát có nội dung tương tự nhất.

Mô hình sẽ được xuất ra file `.pkl` để sử dụng trực tiếp trong ứng dụng thực tế.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

# Tạo thư mục lưu model nếu chưa có
os.makedirs('models', exist_ok=True)

print('✅ Import thư viện thành công!')

✅ Import thư viện thành công!


## 1. Tải và Tiền xử lý Dữ liệu
Sử dụng file `merged_data.csv`. Chúng ta chỉ lấy thông tin duy nhất cho mỗi bài hát (`song_id`) để xây dựng thư viện.

In [2]:
# Đọc dữ liệu
df = pd.read_csv('processed_data/merged_data.csv')
print(f'Kích thước dữ liệu gốc: {df.shape[0]:,} dòng x {df.shape[1]} cột')

# Lọc bỏ các bản ghi trùng lặp bài hát
df = df.drop_duplicates(subset=['song_id']).reset_index(drop=True)
print(f'Kích thước dữ liệu bài hát duy nhất: {df.shape[0]:,} dòng')

df.head(3)

Kích thước dữ liệu gốc: 174,010 dòng x 16 cột
Kích thước dữ liệu bài hát duy nhất: 20,474 dòng


,msno,song_id,target,city,bd,gender,registered_via,registration_init_time,expiration_date,artist_name,language,main_genre,composer,lyricist,song_name,primary_genre
0,a9vIipKPM2RDZc8JC1VFMAzfXwU7dCm20DKvE6Lr5I8=,A7xWQbpX8PCSanLLOALjd7Y3VR9FJwN7w9076bKxYeM=,1,15,34,female,9,20110205,20170915,BIGBANG,31.0,444,TEDDY| G-DRAGON,TEDDY| G-DRAGON,LET’S NOT FALL IN LOVE,444
1,IY77gx4vqofYImTT7heviblox8Xzq7Ng5jlvov0jphM=,wBTWuHbjdjxnG1lQcbqnK4FddV24rUhuyrYLd9c/hmk=,1,12,23,male,9,20130907,20170922,田馥甄 (Hebe),3.0,465,JerryC,徐世珍/吳輝福,小幸運 (A little happiness),465
2,fDGKxxbD6qximlX23UV/eGNy+laemVGA+U3mUZcnQgk=,bTlSHgnwaBMIKHseTFHanJ3Zr4x82ybrGkexpnxDhTY=,1,8,19,male,9,20161113,20170923,Eric 周興哲,3.0,458,Eric Chou,Eric Chou| Yi-Wei Wu,愛情教會我們的事 (What Love Has Taught Us),458


## 2. Xây dựng Cột Content
Tạo đoạn văn bản tổng hợp các thông tin quan trọng. `song_name` được lặp lại 2 lần để tăng trọng số cho tên bài hát, giúp phân biệt tốt hơn giữa các bài hát của cùng một nghệ sĩ.

In [3]:
# Xử lý giá trị bị thiếu
df['song_name'] = df['song_name'].fillna('')
df['artist_name'] = df['artist_name'].fillna('')
df['main_genre'] = df['main_genre'].fillna('')
df['composer'] = df['composer'].fillna('')
df['lyricist'] = df['lyricist'].fillna('')
df['language'] = df['language'].fillna('')

# Nối các thông tin văn bản
df['content'] = (
    df['song_name'].astype(str) + " " +
    df['song_name'].astype(str) + " " +
    df['artist_name'].astype(str) + " " +
    df['main_genre'].astype(str) + " " +
    df['composer'].astype(str) + " " +
    df['lyricist'].astype(str) + " " +
    df['language'].astype(str)
)

print("Ví dụ về Content của một bài hát:")
print(df['content'].iloc[0])

Ví dụ về Content của một bài hát:
LET’S NOT FALL IN LOVE LET’S NOT FALL IN LOVE BIGBANG 444 TEDDY| G-DRAGON TEDDY| G-DRAGON 31.0


## 3. Vector hóa với TF-IDF
Chuyển đổi chuỗi văn bản `content` thành ma trận đặc trưng sử dụng `TfidfVectorizer` với tối đa `10000` features.

In [4]:
# Khởi tạo TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)

# Fit và Transform dữ liệu
tfidf_matrix = tfidf.fit_transform(df['content'])

print(f'Kích thước ma trận TF-IDF: {tfidf_matrix.shape}')

Kích thước ma trận TF-IDF: (20474, 10000)


## 4. Huấn luyện Mô hình Content-based KNN
Sử dụng khoảng cách Cosine trên ma trận TF-IDF để đo lường độ tương đồng giữa các bài hát.

In [5]:
# Khởi tạo mô hình KNN
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)

# Fit mô hình với ma trận TF-IDF
knn_model.fit(tfidf_matrix)

# Tạo Dictionary mapping từ song_id sang index
song_idx = pd.Series(df.index, index=df['song_id']).drop_duplicates()

print('✅ Huấn luyện mô hình KNN Content-based thành công!')

✅ Huấn luyện mô hình KNN Content-based thành công!


## 5. Lưu Mô hình và Dữ liệu (Export to .pkl)
Gói toàn bộ mô hình và các thông tin liên quan thành một object duy nhất và lưu bằng `pickle` để dễ dàng load lên ứng dụng.

In [6]:
# Gom nhóm dữ liệu model
model_package = {
    "tfidf": tfidf,
    "tfidf_matrix": tfidf_matrix,
    "knn_model": knn_model,
    "songs_df": df,
    "song_idx": song_idx
}

# Lưu file model (lưu ở thư mục gốc để app.py dễ dàng load)
root_model_path = "content_based_model.pkl"

with open(root_model_path, "wb") as f:
    pickle.dump(model_package, f)

print(f'💾 Đã lưu model thành công vào {root_model_path}')

💾 Đã lưu model thành công vào content_based_model.pkl


## 6. Xây dựng Hàm Gợi ý Content-based & Đánh giá
Hàm gợi ý sẽ nhận vào `song_id`, tìm các bài hát có content tương tự. Đồng thời tính điểm trung bình (Average Similarity) của top gợi ý để đánh giá chất lượng mô hình (kế thừa logic từ `test_content_base.py`).

In [7]:
def recommend_song_content_based(song_id, top_n=5, verbose=True):
    if song_id not in song_idx:
        if verbose: print("❌ Song ID không tồn tại!")
        return None
        
    idx = int(song_idx[song_id])
    
    if idx >= tfidf_matrix.shape[0]:
        if verbose: print("❌ Index vượt ngoài phạm vi!")
        return None
        
    # Tính khoảng cách và lấy indices (lấy top_n + 5 để bù lại các bài trùng)
    distances, indices = knn_model.kneighbors(tfidf_matrix[idx], n_neighbors=top_n + 5)
    
    rec_indices = indices.flatten()[1:]
    rec_distances = distances.flatten()[1:]
    
    # Lọc bỏ các bài có cosine distance ≈ 0 (chính nó hoặc bài quá giống hệt)
    mask = rec_distances > 1e-6
    valid_indices = rec_indices[mask][:top_n]
    valid_distances = rec_distances[mask][:top_n]
    
    # Lấy thông tin bài hát gốc
    original_song = df.iloc[idx]
    
    if verbose:
        print("\n" + "=" * 50)
        print("🎵 KẾT QUẢ GỢI Ý (CONTENT-BASED)")
        print("=" * 50)
        print(f"Bài hát gốc : {original_song['song_name']}")
        print(f"Ca sĩ       : {original_song['artist_name']}")
        print(f"Thể loại    : {original_song['main_genre']}")
        print("\nCác bài hát gợi ý:\n")
        
    similarity_scores = []
    recommended_songs = []
    
    for i, (dist, row_idx) in enumerate(zip(valid_distances, valid_indices), start=1):
        if row_idx >= len(df): continue
        
        row = df.iloc[row_idx]
        similarity = 1 - dist
        similarity_scores.append(similarity)
        
        recommended_songs.append({
            'song_id': row['song_id'],
            'song_name': row['song_name'],
            'artist_name': row['artist_name'],
            'main_genre': row['main_genre'],
            'similarity_score': round(similarity, 4)
        })
        
        if verbose:
            print(f"{i}. {row['song_name']}")
            print(f"   Ca sĩ         : {row['artist_name']}")
            print(f"   Thể loại      : {row['main_genre']}")
            print(f"   Độ tương đồng : {similarity:.4f}")
            print("-" * 40)
            
    # Đánh giá độ tin cậy của kết quả gợi ý
    if verbose and len(similarity_scores) > 0:
        avg_similarity = np.mean(similarity_scores)
        print("\n" + "=" * 50)
        print("📊 ĐÁNH GIÁ MÔ HÌNH")
        print("=" * 50)
        print(f"Điểm tương đồng trung bình: {avg_similarity:.4f}")
        
        if avg_similarity >= 0.7:
            print("✅ Chất lượng gợi ý: RẤT TỐT (VERY GOOD)")
        elif avg_similarity >= 0.5:
            print("✅ Chất lượng gợi ý: TỐT (GOOD)")
        else:
            print("⚠️ Chất lượng gợi ý: CẦN CẢI THIỆN (NEED IMPROVEMENT)")
            
    return pd.DataFrame(recommended_songs)

## 7. Demo & Kiểm thử Mô hình
Lấy mẫu một vài bài hát trong dataset và thử nghiệm thuật toán.

In [8]:
# Hiển thị một số bài hát mẫu
print("Một số bài hát mẫu trong dataset:")
sample_songs = df[['song_id', 'song_name', 'artist_name']].head(10)
display(sample_songs)

# Test với bài hát đầu tiên
test_song_id = df.iloc[0]['song_id']
recs = recommend_song_content_based(test_song_id, top_n=5)

Một số bài hát mẫu trong dataset:


,song_id,song_name,artist_name
0,A7xWQbpX8PCSanLLOALjd7Y3VR9FJwN7w9076bKxYeM=,LET’S NOT FALL IN LOVE,BIGBANG
1,wBTWuHbjdjxnG1lQcbqnK4FddV24rUhuyrYLd9c/hmk=,小幸運 (A little happiness),田馥甄 (Hebe)
2,bTlSHgnwaBMIKHseTFHanJ3Zr4x82ybrGkexpnxDhTY=,愛情教會我們的事 (What Love Has Taught Us),Eric 周興哲
3,aILZwTNyIlkTuqbWfVhvmUp5Nh1I36iOwhhULcTdl28=,靠岸,林宇中 (Rynn Lim)
4,v/3onppBGoSpGsWb8iaCIO8eX5+iacbH5a4ZUhT7N54=,Alone,Alan Walker
5,U4Of3iMkMaGD0AaimQqKVETejqBi8m6TsM59fMYisKU=,不公平 [插曲 小燕子的心聲],Various Artists
6,cy10N2j2sdY/X4BDUcMu2Iumfz7pV3tqE5iEaup2yGI=,派對動物 (Party Animal),五月天 (Mayday)
7,d5ayexvXscdzmuGxENyY8Uwb4AQaxt0dEcnkAgES7xw=,明天再擱來,玖壹壹
8,gffrcQ9mFCfCwM4Gx7ArXpR8J749p3tfh/tM0chIODo=,Say Yes,Loco & Punch
9,G/4+VCRLpfjQJ4SAwMDcf+W8PTw0eOBRgFvg4fHUOO8=,道聽塗說 (Remembering you),林芯儀 (Shennio Lin)



🎵 KẾT QUẢ GỢI Ý (CONTENT-BASED)
Bài hát gốc : LET’S NOT FALL IN LOVE
Ca sĩ       : BIGBANG
Thể loại    : 444

Các bài hát gợi ý:

1. Let’s not fall in love
   Ca sĩ         : BIGBANG
   Thể loại      : 465
   Độ tương đồng : 0.9870
----------------------------------------
2. 愛情塵埃(LOVE DUST)
   Ca sĩ         : BIGBANG
   Thể loại      : 465
   Độ tương đồng : 0.5815
----------------------------------------
3. BLUE
   Ca sĩ         : BIGBANG
   Thể loại      : 465
   Độ tương đồng : 0.5507
----------------------------------------
4. STILL ALIVE
   Ca sĩ         : BIGBANG
   Thể loại      : 465
   Độ tương đồng : 0.5182
----------------------------------------
5. GIRLFRIEND
   Ca sĩ         : BIGBANG
   Thể loại      : 444
   Độ tương đồng : 0.5138
----------------------------------------

📊 ĐÁNH GIÁ MÔ HÌNH
Điểm tương đồng trung bình: 0.6302
✅ Chất lượng gợi ý: TỐT (GOOD)
